# Notebook 3: Optimizer Comparison Study (M4 — paper centerpiece)

Seven first-order optimizers, hand-rolled, all training the same logistic regression on the heart-disease task with the same mini-batch schedule and the same initialization. We tune each optimizer's learning rate over a small grid, then report the *best-lr* convergence curve so the comparison is fair.

Optimizers covered:
1. SGD (mini-batch)
2. SGD with Polyak momentum
3. SGD with Nesterov accelerated gradient (NAG)
4. AdaGrad
5. RMSprop
6. Adam
7. AdamW (Adam with decoupled weight decay)

Reference: sklearn L-BFGS from notebook 02 (val AUC 0.95197).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, log_loss

SPLITS_DIR = '/content/drive/MyDrive/ECE567_Final/splits'
SEED = 2540
np.random.seed(SEED)

data = np.load(os.path.join(SPLITS_DIR, 'split_preproc.npz'), allow_pickle=True)
X_tr, X_val = data['X_tr'], data['X_val']
y_tr, y_val = data['y_tr'].astype(np.float64), data['y_val'].astype(np.float64)
feature_names = list(data['feature_names'])

with open(os.path.join(SPLITS_DIR, 'results_baselines.json')) as fh:
    baselines = json.load(fh)
LBFGS_AUC = baselines['sklearn_lbfgs']['val_auc']

print('train:', X_tr.shape, 'val:', X_val.shape)
print(f'L-BFGS reference val AUC = {LBFGS_AUC:.5f}')

### Shared loss / gradient utilities

In [ ]:
def sigmoid(z):
    out = np.empty_like(z)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    ez = np.exp(z[~pos])
    out[~pos] = ez / (1.0 + ez)
    return out

def bce_loss(y, p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def predict_proba(X, w, b):
    return sigmoid(X @ w + b)

def grad_logreg(X, y, w, b):
    """Returns (grad_w, grad_b) of mean BCE over the given mini-batch."""
    p = sigmoid(X @ w + b)
    err = p - y
    N = X.shape[0]
    return X.T @ err / N, err.mean()

### Optimizer implementations

Each optimizer is a class with `.step(grad_w, grad_b, lr)` that returns the parameter delta. Stateless w/b updates live in the training loop.

Update rules (paper-ready):
- **SGD**: $\theta \leftarrow \theta - \eta g$.
- **Momentum**: $v \leftarrow \mu v + g$, $\theta \leftarrow \theta - \eta v$.
- **NAG (Sutskever form)**: $v_{k+1} \leftarrow \mu v_k - \eta g_k$, $\theta_{k+1} \leftarrow \theta_k + \mu v_{k+1} - \eta g_k$.
- **AdaGrad**: $G \leftarrow G + g^2$, $\theta \leftarrow \theta - \eta g / (\sqrt{G}+\epsilon)$.
- **RMSprop**: $E \leftarrow \beta E + (1-\beta) g^2$, $\theta \leftarrow \theta - \eta g / (\sqrt{E}+\epsilon)$.
- **Adam**: $m \leftarrow \beta_1 m + (1-\beta_1) g$, $v \leftarrow \beta_2 v + (1-\beta_2) g^2$, $\hat m = m/(1-\beta_1^t)$, $\hat v = v/(1-\beta_2^t)$, $\theta \leftarrow \theta - \eta \hat m / (\sqrt{\hat v}+\epsilon)$.
- **AdamW**: as Adam, plus $\theta \leftarrow \theta - \eta \lambda \theta$ (decoupled L2).

In [ ]:
class SGD:
    name = 'SGD'
    def __init__(self, D):
        pass
    def step(self, gw, gb, lr):
        return -lr * gw, -lr * gb

class Momentum:
    name = 'Momentum'
    def __init__(self, D, mu=0.9):
        self.mu = mu
        self.vw = np.zeros(D); self.vb = 0.0
    def step(self, gw, gb, lr):
        self.vw = self.mu * self.vw + gw
        self.vb = self.mu * self.vb + gb
        return -lr * self.vw, -lr * self.vb

class NAG:
    name = 'NAG'
    def __init__(self, D, mu=0.9):
        self.mu = mu
        self.vw = np.zeros(D); self.vb = 0.0
    def step(self, gw, gb, lr):
        vw_new = self.mu * self.vw - lr * gw
        vb_new = self.mu * self.vb - lr * gb
        dw = self.mu * vw_new - lr * gw
        db = self.mu * vb_new - lr * gb
        self.vw, self.vb = vw_new, vb_new
        return dw, db

class AdaGrad:
    name = 'AdaGrad'
    def __init__(self, D, eps=1e-8):
        self.eps = eps
        self.Gw = np.zeros(D); self.Gb = 0.0
    def step(self, gw, gb, lr):
        self.Gw += gw * gw
        self.Gb += gb * gb
        return -lr * gw / (np.sqrt(self.Gw) + self.eps), -lr * gb / (np.sqrt(self.Gb) + self.eps)

class RMSprop:
    name = 'RMSprop'
    def __init__(self, D, beta=0.9, eps=1e-8):
        self.beta = beta; self.eps = eps
        self.Ew = np.zeros(D); self.Eb = 0.0
    def step(self, gw, gb, lr):
        self.Ew = self.beta * self.Ew + (1 - self.beta) * gw * gw
        self.Eb = self.beta * self.Eb + (1 - self.beta) * gb * gb
        return -lr * gw / (np.sqrt(self.Ew) + self.eps), -lr * gb / (np.sqrt(self.Eb) + self.eps)

class Adam:
    name = 'Adam'
    def __init__(self, D, b1=0.9, b2=0.999, eps=1e-8):
        self.b1, self.b2, self.eps = b1, b2, eps
        self.mw = np.zeros(D); self.vw = np.zeros(D)
        self.mb = 0.0; self.vb = 0.0
        self.t = 0
    def step(self, gw, gb, lr):
        self.t += 1
        self.mw = self.b1 * self.mw + (1 - self.b1) * gw
        self.vw = self.b2 * self.vw + (1 - self.b2) * gw * gw
        self.mb = self.b1 * self.mb + (1 - self.b1) * gb
        self.vb = self.b2 * self.vb + (1 - self.b2) * gb * gb
        bc1 = 1 - self.b1 ** self.t
        bc2 = 1 - self.b2 ** self.t
        mh_w = self.mw / bc1; vh_w = self.vw / bc2
        mh_b = self.mb / bc1; vh_b = self.vb / bc2
        return -lr * mh_w / (np.sqrt(vh_w) + self.eps), -lr * mh_b / (np.sqrt(vh_b) + self.eps)

class AdamW(Adam):
    name = 'AdamW'
    def __init__(self, D, b1=0.9, b2=0.999, eps=1e-8, wd=1e-4):
        super().__init__(D, b1=b1, b2=b2, eps=eps)
        self.wd = wd
    # Adam.step gives the gradient-based delta; the training loop
    # adds the decoupled -lr*wd*theta term once it sees current theta.
    
OPTIMIZERS = [SGD, Momentum, NAG, AdaGrad, RMSprop, Adam, AdamW]

### Mini-batch training loop

In [ ]:
def train(opt_cls, lr, X, y, X_val, y_val,
          epochs=20, batch_size=2048, seed=SEED, wd=1e-4):
    rng = np.random.default_rng(seed)
    N, D = X.shape
    w = np.zeros(D); b = 0.0
    opt = opt_cls(D, wd=wd) if opt_cls is AdamW else opt_cls(D)

    hist = {'epoch': [], 'train_loss': [], 'val_auc': []}
    n_batches = int(np.ceil(N / batch_size))
    for ep in range(epochs):
        idx = rng.permutation(N)
        ep_loss = 0.0
        for bi in range(n_batches):
            batch = idx[bi*batch_size:(bi+1)*batch_size]
            Xb = X[batch]; yb = y[batch]
            gw, gb = grad_logreg(Xb, yb, w, b)
            dw, db = opt.step(gw, gb, lr)
            if opt_cls is AdamW:
                dw = dw - lr * opt.wd * w
            w += dw; b += db
            ep_loss += bce_loss(yb, sigmoid(Xb @ w + b)) * Xb.shape[0]
        ep_loss /= N
        val_auc = roc_auc_score(y_val, predict_proba(X_val, w, b))
        hist['epoch'].append(ep + 1)
        hist['train_loss'].append(ep_loss)
        hist['val_auc'].append(val_auc)
    return w, b, hist

### Learning-rate sweep — pick the best lr per optimizer

In [ ]:
LR_GRID = {
    'SGD'      : [0.05, 0.1, 0.5, 1.0, 3.0],
    'Momentum' : [0.005, 0.01, 0.05, 0.1, 0.5],
    'NAG'      : [0.005, 0.01, 0.05, 0.1, 0.5],
    'AdaGrad'  : [0.05, 0.1, 0.5, 1.0, 3.0],
    'RMSprop'  : [0.0005, 0.001, 0.005, 0.01, 0.05],
    'Adam'     : [0.0005, 0.001, 0.005, 0.01, 0.05],
    'AdamW'    : [0.0005, 0.001, 0.005, 0.01, 0.05],
}
EPOCHS = 20
BATCH  = 2048

sweep_runs = []  # list of dicts
for opt_cls in OPTIMIZERS:
    name = opt_cls.name
    print(f'\n=== {name} ===')
    for lr in LR_GRID[name]:
        t0 = time.time()
        try:
            w, b, hist = train(opt_cls, lr, X_tr, y_tr, X_val, y_val,
                                epochs=EPOCHS, batch_size=BATCH)
            elapsed = time.time() - t0
            final_auc = hist['val_auc'][-1]
            best_auc  = max(hist['val_auc'])
            print(f'  lr={lr:<8g} final_auc={final_auc:.5f}  best_auc={best_auc:.5f}  t={elapsed:.1f}s')
            sweep_runs.append({'optimizer': name, 'lr': lr, 'final_auc': final_auc,
                                'best_auc': best_auc, 'time_s': elapsed, 'hist': hist})
        except (FloatingPointError, ValueError) as e:
            print(f'  lr={lr:<8g} DIVERGED ({type(e).__name__})')
            sweep_runs.append({'optimizer': name, 'lr': lr, 'final_auc': float('nan'),
                                'best_auc': float('nan'), 'time_s': float('nan'), 'hist': None})

In [ ]:
sweep_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'hist'} for r in sweep_runs])
print(sweep_df.pivot_table(index='optimizer', columns='lr', values='best_auc'))

# pick winning lr per optimizer
best = (sweep_df.dropna(subset=['best_auc'])
        .sort_values('best_auc', ascending=False)
        .drop_duplicates('optimizer', keep='first'))
print('\n--- best lr per optimizer ---')
print(best[['optimizer','lr','best_auc','final_auc','time_s']].to_string(index=False))

### Convergence plot — best lr each

In [ ]:
best_runs = {row.optimizer: [r for r in sweep_runs
                              if r['optimizer']==row.optimizer and r['lr']==row.lr][0]
             for row in best.itertuples()}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
colors = plt.cm.tab10(np.linspace(0, 1, len(best_runs)))
for (name, run), c in zip(best_runs.items(), colors):
    h = run['hist']
    axes[0].plot(h['epoch'], h['train_loss'], label=f'{name}  lr={run["lr"]:g}', color=c)
    axes[1].plot(h['epoch'], h['val_auc'],    label=f'{name}  lr={run["lr"]:g}', color=c)

axes[0].set_xlabel('epoch'); axes[0].set_ylabel('train BCE'); axes[0].set_title('Training loss')
axes[0].set_yscale('log'); axes[0].grid(alpha=0.3); axes[0].legend(fontsize=8)

axes[1].axhline(LBFGS_AUC, ls='--', color='black', alpha=0.6,
                label=f'L-BFGS ref = {LBFGS_AUC:.4f}')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('val AUC'); axes[1].set_title('Validation AUC')
axes[1].grid(alpha=0.3); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(SPLITS_DIR, 'fig_optimizer_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.show()

### Final summary table

In [ ]:
summary_rows = []
for name, run in best_runs.items():
    h = run['hist']
    # epoch at which val AUC first reaches within 0.001 of L-BFGS
    threshold = LBFGS_AUC - 0.001
    hit = [e for e, a in zip(h['epoch'], h['val_auc']) if a >= threshold]
    summary_rows.append({
        'optimizer': name,
        'best_lr'  : run['lr'],
        'final_train_loss': h['train_loss'][-1],
        'final_val_auc'   : h['val_auc'][-1],
        'best_val_auc'    : run['best_auc'],
        'epochs_to_match' : hit[0] if hit else None,
        'time_s'          : run['time_s'],
    })
summary = pd.DataFrame(summary_rows)
print(summary.to_string(index=False))

### Save results for the paper

In [ ]:
out = {
    'config': {'epochs': EPOCHS, 'batch_size': BATCH, 'seed': SEED,
                'lr_grid': LR_GRID, 'lbfgs_ref_auc': LBFGS_AUC},
    'sweep' : [{k: v for k, v in r.items() if k != 'hist'} for r in sweep_runs],
    'best'  : summary.to_dict(orient='records'),
    'curves': {name: {'epoch': run['hist']['epoch'],
                       'train_loss': run['hist']['train_loss'],
                       'val_auc': run['hist']['val_auc'],
                       'best_lr': run['lr']}
                for name, run in best_runs.items()},
}
with open(os.path.join(SPLITS_DIR, 'results_optimizers.json'), 'w') as fh:
    json.dump(out, fh, indent=2, default=float)
summary.to_csv(os.path.join(SPLITS_DIR, 'results_optimizers_summary.csv'), index=False)
print('saved results_optimizers.json and results_optimizers_summary.csv')